In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thapaprabin4/fuel-consumption1/Fuel_Consumption.csv


In [2]:
import pandas as pd

file_path = r'/kaggle/input/datasets/thapaprabin4/fuel-consumption1/Fuel_Consumption.csv'

fuel_df = pd.read_csv(file_path)

In [3]:
fuel_df.drop_duplicates(inplace=True)

In [4]:
fuel_df

,Year,MAKE,MODEL,VEHICLE CLASS,ENGINE SIZE,CYLINDERS,TRANSMISSION,FUEL,FUEL CONSUMPTION,COEMISSIONS
0,2000,ACURA,1.6EL,COMPACT,1.6,4,A4,X,10.5,216
1,2000,ACURA,1.6EL,COMPACT,1.6,4,M5,X,9.8,205
2,2000,ACURA,3.2TL,MID-SIZE,3.2,6,AS5,Z,13.7,265
3,2000,ACURA,3.5RL,MID-SIZE,3.5,6,A4,Z,15.0,301
4,2000,ACURA,INTEGRA,SUBCOMPACT,1.8,4,A4,X,11.4,230
...,...,...,...,...,...,...,...,...,...,...
634,2000,VOLVO,V70 AWD TURBO,STATION WAGON - MID-SIZE,2.4,5,A4,Z,14.4,288
635,2000,VOLVO,V70 GLT TURBO,STATION WAGON - MID-SIZE,2.4,5,A4,Z,13.6,274
636,2000,VOLVO,V70 T5 TURBO,STATION WAGON - MID-SIZE,2.3,5,A4,Z,13.9,274
637,2000,VOLVO,V70 T5 TURBO,STATION WAGON - MID-SIZE,2.3,5,M5,Z,13.0,260


In [5]:
# Drop unwanted columns
fuel_df.drop(columns=['Year', 'MODEL'], inplace=True)
fuel_df.sample(4)

,MAKE,VEHICLE CLASS,ENGINE SIZE,CYLINDERS,TRANSMISSION,FUEL,FUEL CONSUMPTION,COEMISSIONS
331,HONDA,SUBCOMPACT,2.2,4,M5,Z,12.0,251
288,GMC,VAN - CARGO,4.3,6,A4,X,17.8,368
458,OLDSMOBILE,MID-SIZE,3.5,6,A4,X,14.7,283
154,DAEWOO,SUBCOMPACT,1.5,4,M5,X,10.3,205


In [6]:
# Class of columns
num_col = ['ENGINE SIZE', 'CYLINDERS', 'FUEL CONSUMPTION']
ord_col = ['VEHICLE CLASS']
nom_col = ['MAKE', 'TRANSMISSION', 'FUEL']
target  = 'COEMISSIONS'

size_order = ["MINICOMPACT", "SUBCOMPACT", "COMPACT", "TWO-SEATER", "STATION WAGON - SMALL",
    "MID-SIZE", "STATION WAGON - MID-SIZE", "MINIVAN", "SUV", "PICKUP TRUCK - SMALL",
    "PICKUP TRUCK - STANDARD", "FULL-SIZE", "VAN - PASSENGER", "VAN - CARGO"]

In [7]:
#Standard scaling for numerical column
from sklearn.preprocessing import StandardScaler

num_scaler= StandardScaler()

num_scaler.fit(fuel_df[num_col])
num_transformed= num_scaler.transform(fuel_df[num_col])

print(num_transformed[:3, :])
print(type(num_transformed))

[[-1.35257952 -1.11042355 -1.27400258]
 [-1.35257952 -1.11042355 -1.48603565]
 [-0.05247295  0.1216638  -0.30470852]]
<class 'numpy.ndarray'>


In [8]:
#Ordinal encoding for ordinal column
from sklearn.preprocessing import OrdinalEncoder

ord_enc=OrdinalEncoder(categories=[size_order])

ord_enc.fit(fuel_df[ord_col])
ord_transform= ord_enc.transform(fuel_df[ord_col])
print(ord_transform[:3])
print(type(ord_transform))

[[2.]
 [2.]
 [5.]]
<class 'numpy.ndarray'>


In [9]:
!pip install -q category_encoders

In [10]:
# Binary encoding for nominal column

from category_encoders import BinaryEncoder

be = BinaryEncoder(nom_col)

be.fit(fuel_df[nom_col])
nom_transformed = be.transform(fuel_df[nom_col]).to_numpy()

print(nom_transformed[:3])
print(type(nom_transformed))

[[0 0 0 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 0 1 0 0 0 1]
 [0 0 0 0 0 1 0 0 1 1 0 1 0]]
<class 'numpy.ndarray'>


In [11]:
import numpy as np

X = np.concat([num_transformed, nom_transformed, ord_transform ], axis=1)
y = fuel_df[target].to_numpy()

print(X.shape)
print(y.shape)

print()
print("Sample Features:\n")
print(X[:3, :])

(638, 17)
(638,)

Sample Features:

[[-1.35257952 -1.11042355 -1.27400258  0.          0.          0.
   0.          0.          1.          0.          0.          0.
   1.          0.          0.          1.          2.        ]
 [-1.35257952 -1.11042355 -1.48603565  0.          0.          0.
   0.          0.          1.          0.          0.          1.
   0.          0.          0.          1.          2.        ]
 [-0.05247295  0.1216638  -0.30470852  0.          0.          0.
   0.          0.          1.          0.          0.          1.
   1.          0.          1.          0.          5.        ]]


In [12]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsRegressor
import numpy as np
model=KNeighborsRegressor(n_neighbors=3,metric="euclidean",weights='distance')
cv_result=cross_val_score(model,X,y,cv=5,scoring="neg_root_mean_squared_error")
print(np.mean(np.abs(cv_result)))

22.666455600847666


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

model=KNeighborsRegressor()
param_grid={
    "n_neighbors":range(1,10),
    "weights":["distance","uniform"],
    "metric":["euclidean","manhattan"]
}

grid_model=GridSearchCV(model,param_grid,cv=15,scoring='neg_root_mean_squared_error')
grid_model.fit(X,y)

GridSearchCV(cv=15, estimator=KNeighborsRegressor(),
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': range(1, 10),
                         'weights': ['distance', 'uniform']},
             scoring='neg_root_mean_squared_error')

In [14]:
best_prams=grid_model.best_params_
print(best_prams)

{'metric': 'manhattan', 'n_neighbors': 6, 'weights': 'distance'}


**MODEL PIPELINE**

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from category_encoders import BinaryEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV 
from sklearn.compose import ColumnTransformer

y=fuel_df['COEMISSIONS']
X=fuel_df.drop(columns='COEMISSIONS')

# split data into respective category 
num_col = ['ENGINE SIZE', 'CYLINDERS', 'FUEL CONSUMPTION']
ord_col = ['VEHICLE CLASS']
nom_col = ['MAKE', 'TRANSMISSION', 'FUEL']
target = 'COEMISSIONS'

size_order = ["MINICOMPACT", "SUBCOMPACT", "COMPACT", "TWO-SEATER", "STATION WAGON - SMALL",
    "MID-SIZE", "STATION WAGON - MID-SIZE", "MINIVAN", "SUV", "PICKUP TRUCK - SMALL",
    "PICKUP TRUCK - STANDARD", "FULL-SIZE", "VAN - PASSENGER", "VAN - CARGO"]

# preprocessing step
# scaling
pre_processor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), num_col),
        ('nominal', BinaryEncoder(), nom_col),
        ('ordinal', OrdinalEncoder(categories=[size_order]), ord_col)
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessing', pre_processor),
    ('kregression', KNeighborsRegressor())  
])


param_grid = {
    "kregression__n_neighbors": range(1, 10),  
    "kregression__weights": ["distance", "uniform"],
    "kregression__metric": ["euclidean", "manhattan"]
}




grid_model = GridSearchCV(pipeline, param_grid, cv=15, scoring='neg_root_mean_squared_error')
grid_model.fit(X, y)
best_params = grid_model.best_params_
print(best_params)

{'kregression__metric': 'manhattan', 'kregression__n_neighbors': 8, 'kregression__weights': 'distance'}
